# Echo of the Inkwell — Colab Generator (optional)

This notebook is an **optional** remote generation path for the Echo of the Inkwell manga studio.

## Important limitations
- **Free Colab may have no GPU** (or a short GPU session). If GPU is unavailable, generation with large open models will be slow or fail — use the local **mock** path or a paid GPU runtime.
- Outputs here are **candidates only**. They are **not** auto-approved into the production pipeline.
- Always record **seeds**, model id, and license notes before copying files back into the local project.
- Review `reports/model-licensing.json` on the local project before any KDP production export.

## 1. Install dependencies

Install Pillow always. Diffusers/torch are **optional** and large — only install when you intend to run a real open model on a GPU runtime.

In [ ]:
# Core deps (lightweight)
%pip install -q pillow pydantic python-dotenv

# Optional heavy stack — uncomment only on a GPU runtime with enough disk:
# %pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
# %pip install -q diffusers transformers accelerate safetensors

import json, os, shutil, zipfile
from datetime import datetime, timezone
from pathlib import Path

try:
    import torch
    HAS_TORCH = True
    GPU = torch.cuda.is_available()
except Exception:
    HAS_TORCH = False
    GPU = False

print(f"torch installed: {HAS_TORCH}")
print(f"CUDA GPU available: {GPU}")
if not GPU:
    print("NOTE: No GPU detected. Prefer mock placeholders or switch Colab runtime to GPU.")

## 2. Load generation manifest

Upload a `generation_manifest.json` from your local project (export from prompts or craft manually), **or** edit the sample below.

Expected keys per job: `page_id`, `positive`, `negative`, `width`, `height`, `seed` (optional), `model` (optional).

In [ ]:
WORKDIR = Path("/content/echo_inkwell")
WORKDIR.mkdir(parents=True, exist_ok=True)
OUT = WORKDIR / "outputs"
OUT.mkdir(exist_ok=True)

SAMPLE_MANIFEST = {
    "project": "Echo of the Inkwell",
    "backend_preference": "auto",  # auto | mock | diffusers
    "model": "black-forest-labs/FLUX.1-schnell",
    "jobs": [
        {
            "page_id": "page_01",
            "story_page": 1,
            "positive": "black and white manga line art, coloring book style, teen boy Kaito at desk, unfinished sketchbook",
            "negative": "photorealistic, color fill, blurry, watermark",
            "width": 768,
            "height": 1024,
            "seed": 42,
        }
    ],
}

manifest_path = Path("/content/generation_manifest.json")
if manifest_path.is_file():
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    print(f"Loaded manifest from {manifest_path}")
else:
    manifest = SAMPLE_MANIFEST
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print("Using SAMPLE_MANIFEST (upload generation_manifest.json to override)")

print(json.dumps({k: manifest[k] for k in manifest if k != "jobs"}, indent=2))
print(f"Jobs: {len(manifest.get('jobs', []))}")

## 3. Generate (open model **or** mock fallback)

If GPU + diffusers are available, the notebook attempts an open model load.
Otherwise it clearly falls back to **Pillow mock line-art** labelled `NON-PRODUCTION TEST`.

**License note:** owner must confirm commercial terms for any real model before KDP production.

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import hashlib

NON_PRODUCTION = "NON-PRODUCTION TEST"
results = []

def mock_generate(prompt, negative, width, height, seed, out_path):
    if seed is None:
        seed = int(hashlib.sha256(prompt.encode()).hexdigest()[:8], 16)
    img = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    m = max(8, min(width, height) // 20)
    draw.rectangle([m, m, width - m, height - m], outline="black", width=3)
    draw.text((m + 4, m + 4), NON_PRODUCTION, fill="black")
    draw.text((m + 4, m + 20), f"seed={seed}", fill="black")
    img.save(out_path, format="PNG")
    return seed, "mock-lineart"

pipe = None
mode = "mock"
pref = str(manifest.get("backend_preference", "auto")).lower()
model_id = manifest.get("model") or "black-forest-labs/FLUX.1-schnell"

if pref != "mock" and GPU and HAS_TORCH:
    try:
        from diffusers import AutoPipelineForText2Image
        print(f"Loading open model: {model_id} (this can take a long time / lots of disk)...")
        pipe = AutoPipelineForText2Image.from_pretrained(model_id, torch_dtype=torch.float16)
        pipe.to("cuda")
        mode = "diffusers"
    except Exception as exc:
        print(f"Open model unavailable ({type(exc).__name__}: {exc})")
        print("Falling back to mock Pillow placeholders — NON-PRODUCTION TEST only.")
        pipe = None
        mode = "mock"
else:
    print("Using mock backend (no GPU / pref=mock / torch missing).")

for job in manifest.get("jobs", []):
    page_id = job["page_id"]
    width = int(job.get("width", 768))
    height = int(job.get("height", 1024))
    seed = job.get("seed")
    out_path = OUT / f"{page_id}.png"
    used_model = model_id

    if mode == "diffusers" and pipe is not None:
        generator = torch.Generator(device="cuda")
        if seed is not None:
            generator = generator.manual_seed(int(seed))
        else:
            seed = int(generator.seed())
        image = pipe(
            prompt=job.get("positive", ""),
            negative_prompt=job.get("negative", ""),
            width=width,
            height=height,
            generator=generator,
        ).images[0]
        image.save(out_path)
    else:
        seed, used_model = mock_generate(
            job.get("positive", ""),
            job.get("negative", ""),
            width,
            height,
            seed,
            out_path,
        )

    meta = {
        "page_id": page_id,
        "story_page": job.get("story_page"),
        "seed": seed,
        "model": used_model,
        "backend": mode,
        "width": width,
        "height": height,
        "output": str(out_path.name),
        "non_production": mode == "mock",
        "label": NON_PRODUCTION if mode == "mock" else None,
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "positive": job.get("positive", ""),
        "negative": job.get("negative", ""),
    }
    (OUT / f"{page_id}.meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
    results.append(meta)
    print(f"OK {page_id} seed={seed} backend={mode}")

(OUT / "batch_results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
print(f"Wrote {len(results)} results to {OUT}")

## 4. Package results + export back to local project

Download the zip, then on your machine:

1. Unzip into a scratch folder.
2. Copy PNGs into `generations/pages/<page_id>/` (or import via a future sync script).
3. Create / update generation `record.json` entries with the seed + model from each `*.meta.json`.
4. Review in Streamlit (`streamlit run app.py`) — **do not auto-approve**.
5. Update `reports/model-licensing.json` with the model you actually used.

In [ ]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
zip_path = WORKDIR / f"echo_inkwell_colab_{stamp}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in OUT.rglob("*"):
        if path.is_file():
            zf.write(path, arcname=str(path.relative_to(WORKDIR)))
    readme = WORKDIR / "EXPORT_README.txt"
    readme.write_text(
        "Echo of the Inkwell Colab export\n"
        f"Created: {stamp}\n"
        "Candidates only — not production-approved.\n"
        "Copy into local generations/ and review in Streamlit.\n",
        encoding="utf-8",
    )
    zf.write(readme, arcname="EXPORT_README.txt")

print(f"Package ready: {zip_path}")
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Not running inside Google Colab UI — zip left on disk for manual download.")